# Home Credit Default Risk — Milestone 1: Baseline Model

**Project goal:** Predict which loan applicants are likely to default, so Home Credit can make better approve/reject decisions — especially for applicants with thin/no credit history.

**Metric:** AUC-ROC (matches the original Kaggle competition metric), given the strong class imbalance (~92% no-default vs ~8% default) in the target.

**This notebook covers:**
1. Load & inspect data
2. Handle missing values
3. Encode categorical features
4. Fix a data quality issue (`DAYS_EMPLOYED` placeholder)
5. Train/validation split
6. Baseline models: Logistic Regression & LightGBM
7. Summary comparison

**Repo structure note:** this notebook is designed to live in the `notebooks/` folder of the
project repo, reading data from `../data/raw/`.


## 1. Load & Inspect Data

In [1]:
import pandas as pd
import numpy as np

# Relative path: assumes this notebook lives in notebooks/, with data in data/raw/ (repo root sibling)
DATA_PATH = "../data/raw/application_train.csv"

df = pd.read_csv(DATA_PATH)
sk_id = df['SK_ID_CURR']  # keep for merging engineered features back in later (Milestone 3)

print("Shape:", df.shape)
print("\nColumn dtype counts:")
print(df.dtypes.value_counts())
print("\nTarget distribution (%):")
print((df['TARGET'].value_counts(normalize=True) * 100).round(2))

Shape: (307511, 122)

Column dtype counts:
float64    65
int64      41
str        16
Name: count, dtype: int64

Target distribution (%):
TARGET
0    91.93
1     8.07
Name: proportion, dtype: float64


**Interpretation:** 307,511 rows, 122 columns. Target is heavily imbalanced (~92% / 8%) —
this means accuracy is a misleading metric, and models/splits need to account for the imbalance
(class weighting, stratified splitting, and AUC as the evaluation metric).

## 2. Handle Missing Values

In [2]:
y = df['TARGET']
X = df.drop(columns=['TARGET', 'SK_ID_CURR'])

missing_frac = X.isnull().mean().sort_values(ascending=False)

# Decision: drop columns with >50% missing, EXCEPT known-important features.
# EXT_SOURCE_1 is kept despite ~56% missing: EXT_SOURCE_2/3 are the strongest
# predictors in this dataset, and EXT_SOURCE_1 is the same type of feature
# (external credit score) -- confirmed later by its feature importance.
keep_despite_missing = ['EXT_SOURCE_1']

cols_to_drop = missing_frac[missing_frac > 0.5].index.tolist()
cols_to_drop = [c for c in cols_to_drop if c not in keep_despite_missing]

X = X.drop(columns=cols_to_drop)

print(f"Dropped {len(cols_to_drop)} columns with >50% missing (kept {keep_despite_missing})")
print("Shape after dropping:", X.shape)

Dropped 40 columns with >50% missing (kept ['EXT_SOURCE_1'])
Shape after dropping: (307511, 80)


**Interpretation:** A blanket ">50% missing → drop" rule would have discarded `EXT_SOURCE_1`,
one of the strongest predictors in the dataset (confirmed later by feature importance and
correlation with TARGET ≈ -0.155, close to EXT_SOURCE_2/3). Worth flagging as a deliberate
override of the default rule, based on domain knowledge.

## 3. Encode Categorical Features

In [3]:
from sklearn.preprocessing import LabelEncoder

cat_cols = X.select_dtypes(include=['object', 'str']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object', 'str']).columns.tolist()

print(f"Categorical columns: {len(cat_cols)}, Numeric columns: {len(num_cols)}")

# Fill missing categorical values with an explicit 'missing' category --
# missingness itself may carry signal (e.g. missing OCCUPATION_TYPE could mean
# unemployed/retired, which is informative, not random noise).
for col in cat_cols:
    X[col] = X[col].fillna('missing')

# Label encode. Note: this imposes an artificial numeric order that doesn't
# exist for nominal categories. Fine for tree models (LightGBM just uses it
# for splits), but a known limitation for Logistic Regression, which treats
# the codes as ordinal.
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le

print("Shape after encoding:", X.shape)
print("Dtypes check (should all be numeric):")
print(X.dtypes.value_counts())

Categorical columns: 13, Numeric columns: 67
Shape after encoding: (307511, 80)
Dtypes check (should all be numeric):
int64      52
float64    28
Name: count, dtype: int64


## 4. Fix Data Quality Issue: `DAYS_EMPLOYED`

**Finding:** `DAYS_EMPLOYED` contains a placeholder value of `365243` (~1000 years) in ~18% of rows.
This lines up almost perfectly with `FLAG_EMP_PHONE == 0` — it's a disguised "not currently employed"
code (e.g. retirees), not a real employment duration. Left as-is, it badly distorts
distance/coefficient-based models (like Logistic Regression).

**Fix:** separate the signal from the placeholder — add a flag column, then replace the
placeholder with a proper `np.nan` (NOT `pd.NA`, which breaks LightGBM by converting the
column to object dtype — a real bug hit during development, noted here so it isn't repeated).

In [4]:
X['DAYS_EMPLOYED_ANOM'] = (X['DAYS_EMPLOYED'] == 365243).astype(int)
X['DAYS_EMPLOYED'] = X['DAYS_EMPLOYED'].replace(365243, np.nan)

print("DAYS_EMPLOYED dtype:", X['DAYS_EMPLOYED'].dtype)  # should be float64
print(f"Anomaly flag positive rate: {X['DAYS_EMPLOYED_ANOM'].mean()*100:.2f}%")

DAYS_EMPLOYED dtype: float64
Anomaly flag positive rate: 18.01%


## 5. Train / Validation Split

In [5]:
from sklearn.model_selection import train_test_split

# Stratified split: preserves the ~92/8 TARGET ratio in both sets, so AUC
# comparisons between train/val (and between model iterations) stay reliable.
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Val shape:", X_val.shape)
print("\nTarget distribution — train:", y_train.value_counts(normalize=True).round(4).to_dict())
print("Target distribution — val:  ", y_val.value_counts(normalize=True).round(4).to_dict())

Train shape: (246008, 81)
Val shape: (61503, 81)

Target distribution — train: {0: 0.9193, 1: 0.0807}
Target distribution — val:   {0: 0.9193, 1: 0.0807}


## 6a. Baseline Model — Logistic Regression

Needs manual imputation (median, computed on train only, applied to val — avoids leakage)
and feature scaling (LogReg is sensitive to feature magnitude).

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

train_medians = X_train.median()
X_train_filled = X_train.fillna(train_medians)
X_val_filled = X_val.fillna(train_medians)  # use TRAIN medians on val -- avoids leakage

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_filled)
X_val_scaled = scaler.transform(X_val_filled)

logreg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
logreg.fit(X_train_scaled, y_train)

val_pred_lr = logreg.predict_proba(X_val_scaled)[:, 1]
auc_lr = roc_auc_score(y_val, val_pred_lr)
print(f"Logistic Regression Validation AUC: {auc_lr:.4f}")

coef_df = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': logreg.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

print("\nTop 10 features by absolute coefficient magnitude:")
print(coef_df.head(10).to_string(index=False))

Logistic Regression Validation AUC: 0.7472

Top 10 features by absolute coefficient magnitude:
            feature  coefficient
    AMT_GOODS_PRICE    -0.990157
         AMT_CREDIT     0.916965
       EXT_SOURCE_3    -0.472838
       EXT_SOURCE_2    -0.393865
    CNT_FAM_MEMBERS    -0.243991
       CNT_CHILDREN     0.206088
       EXT_SOURCE_1    -0.185047
        CODE_GENDER     0.177129
      DAYS_EMPLOYED     0.171249
NAME_EDUCATION_TYPE     0.151939


**Interpretation:** After fixing `DAYS_EMPLOYED`, the top coefficients are dominated by
`AMT_GOODS_PRICE`, `AMT_CREDIT`, and the `EXT_SOURCE` scores — sensible drivers for a lending
model. Before the fix, `DAYS_EMPLOYED`/`FLAG_EMP_PHONE` had implausibly large coefficients
(~7.5) due to the placeholder value; AUC barely changed, but the model's coefficients became
meaningfully more trustworthy.

## 6b. Baseline Model — LightGBM

No manual imputation or scaling needed — LightGBM handles missing values natively (learns
the best split direction for NaNs) and tree splits don't care about feature scale.

In [7]:
import lightgbm as lgb

lgb_train = lgb.Dataset(X_train, y_train)
lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)

params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'is_unbalance': True,   # compensates for the 92/8 imbalance, similar role to class_weight in LogReg
    'verbosity': -1,
    'seed': 42
}

model_lgb = lgb.train(
    params, lgb_train,
    valid_sets=[lgb_val],
    num_boost_round=200,
    callbacks=[lgb.early_stopping(20), lgb.log_evaluation(0)]
)

val_pred_lgb = model_lgb.predict(X_val, num_iteration=model_lgb.best_iteration)
auc_lgb = roc_auc_score(y_val, val_pred_lgb)
print(f"LightGBM Validation AUC: {auc_lgb:.4f}")
print(f"Best iteration: {model_lgb.best_iteration}")

importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model_lgb.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

print("\nTop 10 features by importance (gain):")
print(importance.head(10).to_string(index=False))

Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[171]	valid_0's auc: 0.761933
LightGBM Validation AUC: 0.7619
Best iteration: 171

Top 10 features by importance (gain):
            feature    importance
       EXT_SOURCE_3 201679.566411
       EXT_SOURCE_2 152710.905025
       EXT_SOURCE_1  66457.644361
      DAYS_EMPLOYED  42426.502665
         AMT_CREDIT  32496.576123
         DAYS_BIRTH  25913.790261
        AMT_ANNUITY  25289.808619
    AMT_GOODS_PRICE  24973.222741
NAME_EDUCATION_TYPE  16886.963947
    DAYS_ID_PUBLISH  16021.337328


## 7. Summary

In [8]:
summary = pd.DataFrame({
    'Model': ['Logistic Regression', 'LightGBM'],
    'Validation AUC': [auc_lr, auc_lgb]
})
print(summary.to_string(index=False))

              Model  Validation AUC
Logistic Regression        0.747189
           LightGBM        0.761933


**Milestone 1 status: baseline established.**

Confirmed results: Logistic Regression AUC ≈ 0.7472, LightGBM AUC ≈ 0.7619.

Next steps (Milestone 3): error analysis, feature engineering from `bureau.csv` and
`installments_payments.csv`, and a third algorithm with cross-validation and hyperparameter
tuning. See `milestone3_error_analysis.ipynb`.
